In [6]:
"""
run_analysis.py — 통합 시장 분석 스크립트 (미장 + 국장)
========================================================
[사용법]
  python run_analysis.py --market US_ALL               # 미국 전체
  python run_analysis.py --market US_ALL --tickers TSLA NVDA  # 특정 종목
  python run_analysis.py --market KR_ALL               # 한국 전체
  python run_analysis.py --market KR_ALL --tickers 005930 035420  # 삼성전자, NAVER

[개선 사항]
  1. 미장/국장 통합 (MARKET 파라미터로 분기)
  2. strategies.py의 8개 전략 적용
  3. GICS 기반 정확한 섹터 분류
  4. ATR 기반 동적 손절/타겟 (Regime 반영)
  5. Optuna 최적화 선택적 적용 (--optuna 플래그)
  6. 매도 신호 추가
  7. 전략별 보유기간/스타일 표시
"""

import pandas as pd
import numpy as np
import os
import glob
from tqdm import tqdm
from datetime import datetime
import re
import argparse
import warnings
from sector_map import get_sector_etf, get_status_for_stock, get_hts_sector

warnings.filterwarnings('ignore')

# ── 모듈 임포트 ──
from indicators_v2 import (
    calculate_indicators, check_and_fix_columns,
    analyze_volume_profile, check_immediate_hurdle,
    calculate_dynamic_levels
)
from regime_v2 import add_market_regime, detect_regime_transition, get_trading_params
from strategies import (
    apply_all_strategies, get_active_strategies,
    calculate_strategy_levels, check_exit_signals,
    STRATEGY_CONFIG
)
from sector_map import get_sector_etf, get_status_for_stock

# ============================================================
# 설정
# ============================================================

TODAY = datetime.now().strftime('%Y-%m-%d')


def get_config(market):
    """시장별 설정 반환"""
    if market.startswith('US'):
        return {
            'raw_dir': f'./Raw_Data/{market}',
            'result_dir': f'./Results/{market}/{TODAY}',
            'etf_file': 'nasdaq_etf_list.csv',
            'sector_map_file': f'./Raw_Data/{market}/sector_map.csv',
            'sector_result_file': f'./Results/Sector_Analysis/{TODAY}/Sector_Status_Summary.csv',
            'backtest_file': f'./Results/{market}/Backtest/Backtest_Sector_Summary.csv',
            'earnings_file': 'earnings_calendar.csv',
            'currency': '$',
            'code_col': 'Symbol',  # US는 Symbol
        }
    else:  # KR
        return {
            'raw_dir': f'./Raw_Data/{market}',
            'result_dir': f'./Results/{market}/{TODAY}',
            'etf_file': 'kr_etf_list.csv',
            'sector_map_file': f'./Raw_Data/{market}/sector_map.csv',
            'sector_result_file': f'./Results/Sector_Analysis_KR/{TODAY}/Sector_Status_Summary.csv',
            'backtest_file': f'./Results/{market}/Backtest/Backtest_Sector_Summary.csv',
            'earnings_file': 'kr_earnings_calendar.csv',
            'currency': '₩',
            'code_col': 'Code',  # KR는 Code
        }


# ============================================================
# 헬퍼 함수
# ============================================================

def remove_emojis(text):
    if not isinstance(text, str):
        return str(text)
    return re.sub(r'[^\uac00-\ud7a3a-zA-Z0-9\s\(\)\-\_\.]', '', text).strip()


def load_etf_mapping(csv_path):
    if not os.path.exists(csv_path):
        return {}
    try:
        df = pd.read_csv(csv_path)
        if 'Underlying' in df.columns:
            df['Underlying'] = df['Underlying'].str.upper()
            df = df.drop_duplicates(subset=['Underlying'], keep='first')
            return df.set_index('Underlying').to_dict('index')
        return {}
    except Exception:
        return {}


def load_sector_map(csv_path):
    if not os.path.exists(csv_path):
        return {}
    try:
        df = pd.read_csv(csv_path, dtype={'Code': str})
        return df.set_index('Code').to_dict('index')
    except Exception:
        return {}


def load_sector_status(path):
    if not os.path.exists(path):
        return {}
    try:
        df = pd.read_csv(path)
        return df.set_index('Ticker')['Status'].to_dict()
    except Exception:
        return {}


def load_backtest_stats(path):
    if not os.path.exists(path):
        return {}
    try:
        df = pd.read_csv(path)
        if 'Sector' in df.columns:
            return df.set_index('Sector')[['Win_Rate(%)', 'Avg_Duration']].to_dict('index')
        return {}
    except Exception:
        return {}


def calculate_d_day(earnings_date_str):
    if pd.isna(earnings_date_str) or str(earnings_date_str) in ['-', 'N/A', 'nan']:
        return '-', '-'
    try:
        target_date = datetime.strptime(str(earnings_date_str).split()[0], '%Y-%m-%d')
        delta = (target_date - datetime.now()).days + 1
        if delta < 0:
            return earnings_date_str, '지남'
        elif delta == 0:
            return earnings_date_str, 'D-Day'
        else:
            return earnings_date_str, f'D-{delta}'
    except Exception:
        return earnings_date_str, '-'


# ============================================================
# Optuna 최적화 (선택적)
# ============================================================

def optimize_with_optuna(df_full, n_trials=50):
    """
    Regime 기반 Optuna 최적화 (기존 코드 개선)
    — 선택적으로 사용 (시간이 오래 걸리므로)
    """
    try:
        import optuna
        optuna.logging.set_verbosity(optuna.logging.WARNING)
    except ImportError:
        return None

    if 'Regime' not in df_full.columns:
        df_full = add_market_regime(df_full)

    current_regime = int(df_full['Regime'].iloc[-1])
    history = df_full[df_full['Regime'] == current_regime].copy()

    if len(history) < 30:
        history = df_full.iloc[-120:].copy()

    def objective(trial):
        tp_pct = trial.suggest_float('take_profit', 0.02, 0.15)
        sl_pct = trial.suggest_float('stop_loss', 0.01, 0.08)
        signal_th = trial.suggest_int('signal_threshold', 1, 4)

        df = history.copy()
        if 'Composite_Score' not in df.columns:
            return -100

        entry_mask = df['Composite_Score'] >= signal_th
        idx_list = np.where(entry_mask)[0]

        if len(idx_list) < 5:
            return -10

        total_pnl = 0
        trades = 0
        for i in idx_list:
            if i >= len(df) - 5:
                continue
            entry = df['Open'].iloc[i + 1]
            target = entry * (1 + tp_pct)
            stop = entry * (1 - sl_pct)

            future = df.iloc[i + 1: i + 6]  # 5일 홀딩
            if future['Low'].min() <= stop:
                total_pnl -= sl_pct
            elif future['High'].max() >= target:
                total_pnl += tp_pct
            else:
                total_pnl += (future['Close'].iloc[-1] - entry) / entry
            trades += 1

        return total_pnl

    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=n_trials)

    best = study.best_params
    return {
        'optimized_tp_pct': best['take_profit'],
        'optimized_sl_pct': best['stop_loss'],
        'signal_threshold': best['signal_threshold'],
        'regime': current_regime,
    }


# ============================================================
# 메인 분석 함수
# ============================================================

def run_analysis(market='US_ALL', target_tickers=None, use_optuna=False):
    """
    메인 분석 실행
    
    Parameters:
    -----------
    market : str — 'US_ALL' 또는 'KR_ALL'
    target_tickers : list or None — 특정 종목만 분석
    use_optuna : bool — Optuna 최적화 사용 여부
    """
    cfg = get_config(market)
    os.makedirs(cfg['result_dir'], exist_ok=True)

    files = glob.glob(f"{cfg['raw_dir']}/*.csv")
    files = [f for f in files if 'sector_map' not in f]

    # 종목 필터링
    if target_tickers:
        print(f"🔍 지정된 {len(target_tickers)}개 종목만 분석: {target_tickers}")
        filtered = []
        for f in files:
            basename = os.path.basename(f).split('_')[0]
            if basename in target_tickers:
                filtered.append(f)
        files = filtered

    if not files:
        print("❌ 분석할 파일이 없습니다.")
        return

    print(f"🚀 [{market}] 분석 시작 (총 {len(files)}개)")
    if use_optuna:
        print("   ⏳ Optuna 최적화 활성화 — 시간이 걸립니다.")

    # 외부 데이터 로드
    etf_mapper = load_etf_mapping(cfg['etf_file'])
    sector_mapper = load_sector_map(cfg['sector_map_file'])
    sector_status = load_sector_status(cfg['sector_result_file'])
    backtest_stats = load_backtest_stats(cfg['backtest_file'])
    earnings_map = {}
    if os.path.exists(cfg['earnings_file']):
        try:
            earnings_map = pd.read_csv(cfg['earnings_file']).set_index('Ticker')['Next_Earnings'].to_dict()
        except Exception:
            pass

    results = []
    success = 0

    for file_path in tqdm(files):
        try:
            filename = os.path.basename(file_path).replace('.csv', '')
            parts = filename.split('_', 1)
            code = parts[0]
            name = parts[1] if len(parts) > 1 else code

            # ── 1. 데이터 로드 ──
            df = pd.read_csv(file_path)
            if 'Date' in df.columns:
                df['Date'] = pd.to_datetime(df['Date'])
                df.set_index('Date', inplace=True)
                df.sort_index(inplace=True)
            df = check_and_fix_columns(df)
            if df is None:
                continue

            # ── 2. 지표 계산 ──
            df = calculate_indicators(df)
            if df is None or len(df) < 60:
                continue

            # ── 3. Regime 계산 ──
            df = add_market_regime(df)
            current_regime = int(df['Regime'].iloc[-1])
            regime_info = detect_regime_transition(df)

            # ── 4. 전략 적용 ──
            df = apply_all_strategies(df, regime=current_regime)
            df = check_exit_signals(df)
            if df is None:
                continue

            # ── 5. Optuna (선택적) ──
            opt_result = None
            if use_optuna:
                opt_result = optimize_with_optuna(df)

            # ── 6. 매물대 분석 ──
            vp = analyze_volume_profile(df, period=120)
            hurdle = check_immediate_hurdle(df, lookback=40)

    # ── 7. 섹터/ETF 매핑 ──
            sec_info = sector_mapper.get(code, {})
            industry = sec_info.get('Industry', '-')
            sector_name = sec_info.get('Sector', 'Unknown')
            
            # GICS 기반 섹터 ETF 조회 (기존 로직 유지 - 백테스트/ETF 참조용)
            sector_etf, sector_kr = get_sector_etf(industry)
            if sector_etf == '-':
                sector_etf, sector_kr = get_sector_etf(sector_name)
            
            current_sec_status = sector_status.get(sector_etf, '-')

            # ✨ HTS 스타일 한국어 직관적 섹터 변환 추가
            hts_sector = get_hts_sector(industry, sector_name)

            # 실적 D-Day
            clean_date, d_day_str = calculate_d_day(earnings_map.get(code, '-'))
            # 레버리지 ETF
            bull, bear = '-', '-'
            if code in etf_mapper:
                bull = etf_mapper[code].get('Bull_ETF', '-')
                bear = etf_mapper[code].get('Bear_ETF', '-')

            # ── 8. 마지막 행 기준 결과 추출 ──
            last = df.iloc[-1]

            # 활성 전략 분석
            active_strats = get_active_strategies(last)
            strat_names = [s['name'] for s in active_strats]
            strat_styles = list(set(s['style'] for s in active_strats))

            # 손절/타겟 계산 (전략 기반 또는 ATR 기반)
            if active_strats:
                levels = calculate_strategy_levels(last, active_strats)
            else:
                levels = calculate_dynamic_levels(df, current_regime)
                if levels is None:
                    levels = {'target': None, 'stop': None, 'target_pct': None, 'stop_pct': None, 'rr_ratio': None}

            # Optuna 결과 오버라이드
            if opt_result:
                opt_target = round(last['Close'] * (1 + opt_result['optimized_tp_pct']), 2)
                opt_stop = round(last['Close'] * (1 - opt_result['optimized_sl_pct']), 2)
                levels['target'] = opt_target
                levels['stop'] = opt_stop
                levels['target_pct'] = round(opt_result['optimized_tp_pct'] * 100, 2)
                levels['stop_pct'] = round(opt_result['optimized_sl_pct'] * 100, 2)

            # ── 매수 패턴 메시지 ──
            buy_msg = []
            if last.get('Candle_Engulfing'):
                buy_msg.append('상승장악')
            if last.get('Vol_Pump'):
                buy_msg.append('거래량폭발')
            if last.get('Candle_Hammer'):
                buy_msg.append('망치형')
            if last.get('MA20_Bounce'):
                buy_msg.append('20일선지지')
            if last.get('Candle_MorningStar'):
                buy_msg.append('모닝스타')
            if last.get('Bullish_Div'):
                buy_msg.append('상승다이버전스')

            # 활성 전략 추가
            for sn in strat_names:
                buy_msg.append(sn)

            # 경고 메시지
            warn_msg = []
            if last.get('MA20_Break'):
                warn_msg.append('20일선붕괴')
            if last.get('Bearish_Div'):
                warn_msg.append('하락다이버전스')
            if last.get('Candle_BearEngulfing'):
                warn_msg.append('하락장악')
            if last.get('Exit_Signal'):
                warn_msg.append('매도신호')

            # ── 최종 결과 구성 ──
            cur = cfg['currency']
            row_data = {
                'Date': last.name.strftime('%Y-%m-%d') if hasattr(last.name, 'strftime') else str(last.name),
                'Code': code,
                'Name': name,
                
                # ✨ HTS 스타일로 교체
                'Sector': hts_sector, 
                'Sector_ETF': sector_etf,
                'Sector_Status': current_sec_status,
                'Industry': industry,
                # 실적
                'Earnings_Date': clean_date,
                'D-Day': d_day_str,
                
                # 가격
                'Close': round(last['Close'], 2),
                
                # Regime
                'Regime': current_regime,
                'Regime_Trend': regime_info['transition'],
                
                # 신호 (기존 호환)
                'Signal_Count': int(last.get('Signal_Count', 0)),
                'Composite_Score': round(float(last.get('Composite_Score', 0)), 2),
                'Buy_Signal': bool(last.get('Final_Buy', False)),
                'Exit_Signal': bool(last.get('Exit_Signal', False)),
                
                # 전략 상세
                'Active_Strategies': ', '.join(strat_names) if strat_names else '-',
                'Style': ', '.join(strat_styles) if strat_styles else '-',
                'Buy_Pattern': ', '.join(buy_msg) if buy_msg else '-',
                'Warning': ', '.join(warn_msg) if warn_msg else '-',
                
                # 레버리지 ETF
                'Bull_ETF': bull,
                'Bear_ETF': bear,
                
                # 타겟/손절 (ATR 기반 동적 계산)
                f'Target({cur})': levels.get('target') or levels.get('target_price'),
                f'Stop({cur})': levels.get('stop') or levels.get('stop_price'),
                'Target(%)': levels.get('target_pct'),
                'Stop(%)': levels.get('stop_pct'),
                'R:R': levels.get('rr_ratio'),
                
                # 매물대
                'Hurdle_Score': round(hurdle['Hurdle_Score'], 3) if hurdle else 999,
                'SR_Ratio': round(hurdle['SR_Ratio'], 2) if hurdle else 0,
                
                # 백테스트 참고
                'BT_WinRate(%)': backtest_stats.get(sector_name, {}).get('Win_Rate(%)', '-'),
                'BT_AvgDuration': backtest_stats.get(sector_name, {}).get('Avg_Duration', '-'),
                
                # 개별 전략 True/False
                'Trend_Up': bool(last.get('Trend_Up', False)),
                'Momentum': bool(last.get('Momentum', False)),
                'Oversold': bool(last.get('Oversold', False)),
                'Vol_Explode': bool(last.get('Vol_Explode', False)),
                'Vol_Pump': bool(last.get('Vol_Pump', False)),
                'Candle_Buy': bool(last.get('Candle_Buy', False)),
                'MA20_Bounce': bool(last.get('MA20_Bounce', False)),
                'MA20_Break': bool(last.get('MA20_Break', False)),
            }
            results.append(row_data)
            success += 1

        except Exception as e:
            # 디버깅 시 주석 해제:
            # print(f"Error {file_path}: {e}")
            continue

    # ── 결과 저장 ──
    if results:
        all_df = pd.DataFrame(results)
        all_df = all_df.drop_duplicates(subset=['Code'], keep='first')
        all_df = all_df.sort_values(
            by=['Buy_Signal', 'Composite_Score', 'Signal_Count'],
            ascending=[False, False, False]
        )

        all_file = f"{cfg['result_dir']}/{market}_{TODAY}_All_Stocks.csv"
        buy_file = f"{cfg['result_dir']}/{market}_{TODAY}_Buy_Signals.csv"
        exit_file = f"{cfg['result_dir']}/{market}_{TODAY}_Exit_Signals.csv"

        all_df.to_csv(all_file, index=False, encoding='utf-8-sig')

        buy_df = all_df[all_df['Buy_Signal'] == True]
        buy_df.to_csv(buy_file, index=False, encoding='utf-8-sig')

        exit_df = all_df[all_df['Exit_Signal'] == True]
        exit_df.to_csv(exit_file, index=False, encoding='utf-8-sig')

        print(f"\n✅ 분석 완료! (총 {success}개 처리)")
        print(f"🔥 매수 포착: {len(buy_df)}개")
        print(f"⚠️ 매도 신호: {len(exit_df)}개")
        print(f"📂 결과: {buy_file}")

        # 텔레그램 전송
        try:
            import telegram_msg
            msg = f"🚀 **[{market}] {TODAY} 분석 완료!**\n\n"
            msg += f"🔥 매수: {len(buy_df)}개 | ⚠️ 매도: {len(exit_df)}개\n"
            if not buy_df.empty:
                top5 = buy_df.head(5)
                msg += "\n**Top 5:**\n"
                for _, r in top5.iterrows():
                    msg += f"• {r['Code']} ({r['Composite_Score']}점) "
                    msg += f"| {r['Active_Strategies']}\n"
            telegram_msg.send_message(msg)
            telegram_msg.send_file(buy_file, caption=f"🔥 {market} 매수 리스트")
        except Exception:
            pass
    else:
        print("❌ 결과 데이터가 없습니다.")


# ============================================================
# CLI 지원
# ============================================================
if __name__ == '__main__':
    parser = argparse.ArgumentParser(description='시장 분석 스크립트')
    parser.add_argument('--market', type=str, default='KR_ALL',
                        choices=['US_ALL', 'KR_ALL'],
                        help='분석 대상 시장 (US_ALL / KR_ALL)')
    parser.add_argument('--tickers', nargs='*', default=None,
                        help='특정 종목만 분석 (예: TSLA NVDA)')
    parser.add_argument('--optuna', action='store_true',
                        help='Optuna 최적화 사용')
    
    # 🔴 수정된 부분: parse_args() 대신 parse_known_args()를 사용합니다.
    # 이렇게 하면 노트북이 전달하는 --f 인자를 unknown으로 빼고 에러를 내지 않습니다.
    args, unknown = parser.parse_known_args()
    
    run_analysis(
        market=args.market,
        target_tickers=args.tickers,
        use_optuna=args.optuna,
    )
# if __name__ == '__main__':
#     parser = argparse.ArgumentParser(description='시장 분석 스크립트')
#     parser.add_argument('--market', type=str, default='US_ALL',
#                         choices=['US_ALL', 'KR_ALL'],
#                         help='분석 대상 시장 (US_ALL / KR_ALL)')
#     parser.add_argument('--tickers', nargs='*', default=None,
#                         help='특정 종목만 분석 (예: TSLA NVDA)')
#     parser.add_argument('--optuna', action='store_true',
#                         help='Optuna 최적화 사용')
    
#     args = parser.parse_args()
#     run_analysis(
#         market=args.market,
#         target_tickers=args.tickers,
#         use_optuna=args.optuna,
#     )


🚀 [KR_ALL] 분석 시작 (총 2882개)


100%|██████████| 2882/2882 [02:22<00:00, 20.21it/s]


PermissionError: [Errno 13] Permission denied: './Results/KR_ALL/2026-02-23/KR_ALL_2026-02-23_Buy_Signals.csv'

In [ ]:
"""
run_analysis.py — 통합 시장 분석 스크립트 (미장 + 국장)
========================================================
[사용법]
  python run_analysis.py --market US_ALL               # 미국 전체
  python run_analysis.py --market US_ALL --tickers TSLA NVDA  # 특정 종목
  python run_analysis.py --market KR_ALL               # 한국 전체
  python run_analysis.py --market KR_ALL --tickers 005930 035420  # 삼성전자, NAVER

[개선 사항]
  1. 미장/국장 통합 (MARKET 파라미터로 분기)
  2. strategies.py의 8개 전략 적용
  3. GICS 기반 정확한 섹터 분류
  4. ATR 기반 동적 손절/타겟 (Regime 반영)
  5. Optuna 최적화 선택적 적용 (--optuna 플래그)
  6. 매도 신호 추가
  7. 전략별 보유기간/스타일 표시
"""

import pandas as pd
import numpy as np
import os
import glob
from tqdm import tqdm
from datetime import datetime
import re
import argparse
import warnings
from sector_map import get_sector_etf, get_status_for_stock, get_hts_sector

warnings.filterwarnings('ignore')

# ── 모듈 임포트 ──
from indicators_v2 import (
    calculate_indicators, check_and_fix_columns,
    analyze_volume_profile, check_immediate_hurdle,
    calculate_dynamic_levels
)
from regime_v2 import add_market_regime, detect_regime_transition, get_trading_params
from strategies import (
    apply_all_strategies, get_active_strategies,
    calculate_strategy_levels, check_exit_signals,
    STRATEGY_CONFIG
)
from sector_map import get_sector_etf, get_status_for_stock

# ============================================================
# 설정
# ============================================================

TODAY = datetime.now().strftime('%Y-%m-%d')


def get_config(market):
    """시장별 설정 반환"""
    if market.startswith('US'):
        return {
            'raw_dir': f'./Raw_Data/{market}',
            'result_dir': f'./Results/{market}/{TODAY}',
            'etf_file': 'nasdaq_etf_list.csv',
            'sector_map_file': f'./Raw_Data/{market}/sector_map.csv',
            'sector_result_file': f'./Results/Sector_Analysis/{TODAY}/Sector_Status_Summary.csv',
            'backtest_file': f'./Results/{market}/Backtest/Backtest_Sector_Summary.csv',
            'earnings_file': 'earnings_calendar.csv',
            'currency': '$',
            'code_col': 'Symbol',  # US는 Symbol
        }
    else:  # KR
        return {
            'raw_dir': f'./Raw_Data/{market}',
            'result_dir': f'./Results/{market}/{TODAY}',
            'etf_file': 'kr_etf_list.csv',
            'sector_map_file': f'./Raw_Data/{market}/sector_map.csv',
            'sector_result_file': f'./Results/Sector_Analysis_KR/{TODAY}/Sector_Status_Summary.csv',
            'backtest_file': f'./Results/{market}/Backtest/Backtest_Sector_Summary.csv',
            'earnings_file': 'kr_earnings_calendar.csv',
            'currency': '₩',
            'code_col': 'Code',  # KR는 Code
        }


# ============================================================
# 헬퍼 함수
# ============================================================

def remove_emojis(text):
    if not isinstance(text, str):
        return str(text)
    return re.sub(r'[^\uac00-\ud7a3a-zA-Z0-9\s\(\)\-\_\.]', '', text).strip()


def load_etf_mapping(csv_path):
    if not os.path.exists(csv_path):
        return {}
    try:
        df = pd.read_csv(csv_path)
        if 'Underlying' in df.columns:
            df['Underlying'] = df['Underlying'].str.upper()
            df = df.drop_duplicates(subset=['Underlying'], keep='first')
            return df.set_index('Underlying').to_dict('index')
        return {}
    except Exception:
        return {}


def load_sector_map(csv_path):
    if not os.path.exists(csv_path):
        return {}
    try:
        df = pd.read_csv(csv_path, dtype={'Code': str})
        return df.set_index('Code').to_dict('index')
    except Exception:
        return {}


def load_sector_status(path):
    if not os.path.exists(path):
        return {}
    try:
        df = pd.read_csv(path)
        return df.set_index('Ticker')['Status'].to_dict()
    except Exception:
        return {}


def load_backtest_stats(path):
    if not os.path.exists(path):
        return {}
    try:
        df = pd.read_csv(path)
        if 'Sector' in df.columns:
            return df.set_index('Sector')[['Win_Rate(%)', 'Avg_Duration']].to_dict('index')
        return {}
    except Exception:
        return {}


def calculate_d_day(earnings_date_str):
    if pd.isna(earnings_date_str) or str(earnings_date_str) in ['-', 'N/A', 'nan']:
        return '-', '-'
    try:
        target_date = datetime.strptime(str(earnings_date_str).split()[0], '%Y-%m-%d')
        delta = (target_date - datetime.now()).days + 1
        if delta < 0:
            return earnings_date_str, '지남'
        elif delta == 0:
            return earnings_date_str, 'D-Day'
        else:
            return earnings_date_str, f'D-{delta}'
    except Exception:
        return earnings_date_str, '-'


# ============================================================
# Optuna 최적화 (선택적)
# ============================================================

def optimize_with_optuna(df_full, n_trials=50):
    """
    Regime 기반 Optuna 최적화 (기존 코드 개선)
    — 선택적으로 사용 (시간이 오래 걸리므로)
    """
    try:
        import optuna
        optuna.logging.set_verbosity(optuna.logging.WARNING)
    except ImportError:
        return None

    if 'Regime' not in df_full.columns:
        df_full = add_market_regime(df_full)

    current_regime = int(df_full['Regime'].iloc[-1])
    history = df_full[df_full['Regime'] == current_regime].copy()

    if len(history) < 30:
        history = df_full.iloc[-120:].copy()

    def objective(trial):
        tp_pct = trial.suggest_float('take_profit', 0.02, 0.15)
        sl_pct = trial.suggest_float('stop_loss', 0.01, 0.08)
        signal_th = trial.suggest_int('signal_threshold', 1, 4)

        df = history.copy()
        if 'Composite_Score' not in df.columns:
            return -100

        entry_mask = df['Composite_Score'] >= signal_th
        idx_list = np.where(entry_mask)[0]

        if len(idx_list) < 5:
            return -10

        total_pnl = 0
        trades = 0
        for i in idx_list:
            if i >= len(df) - 5:
                continue
            entry = df['Open'].iloc[i + 1]
            target = entry * (1 + tp_pct)
            stop = entry * (1 - sl_pct)

            future = df.iloc[i + 1: i + 6]  # 5일 홀딩
            if future['Low'].min() <= stop:
                total_pnl -= sl_pct
            elif future['High'].max() >= target:
                total_pnl += tp_pct
            else:
                total_pnl += (future['Close'].iloc[-1] - entry) / entry
            trades += 1

        return total_pnl

    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=n_trials)

    best = study.best_params
    return {
        'optimized_tp_pct': best['take_profit'],
        'optimized_sl_pct': best['stop_loss'],
        'signal_threshold': best['signal_threshold'],
        'regime': current_regime,
    }


# ============================================================
# 메인 분석 함수
# ============================================================

def run_analysis(market='US_ALL', target_tickers=None, use_optuna=False):
    """
    메인 분석 실행
    
    Parameters:
    -----------
    market : str — 'US_ALL' 또는 'KR_ALL'
    target_tickers : list or None — 특정 종목만 분석
    use_optuna : bool — Optuna 최적화 사용 여부
    """
    cfg = get_config(market)
    os.makedirs(cfg['result_dir'], exist_ok=True)

    files = glob.glob(f"{cfg['raw_dir']}/*.csv")
    files = [f for f in files if 'sector_map' not in f]

    # 종목 필터링
    if target_tickers:
        print(f"🔍 지정된 {len(target_tickers)}개 종목만 분석: {target_tickers}")
        filtered = []
        for f in files:
            basename = os.path.basename(f).split('_')[0]
            if basename in target_tickers:
                filtered.append(f)
        files = filtered

    if not files:
        print("❌ 분석할 파일이 없습니다.")
        return

    print(f"🚀 [{market}] 분석 시작 (총 {len(files)}개)")
    if use_optuna:
        print("   ⏳ Optuna 최적화 활성화 — 시간이 걸립니다.")

    # 외부 데이터 로드
    etf_mapper = load_etf_mapping(cfg['etf_file'])
    sector_mapper = load_sector_map(cfg['sector_map_file'])
    sector_status = load_sector_status(cfg['sector_result_file'])
    backtest_stats = load_backtest_stats(cfg['backtest_file'])
    earnings_map = {}
    if os.path.exists(cfg['earnings_file']):
        try:
            earnings_map = pd.read_csv(cfg['earnings_file']).set_index('Ticker')['Next_Earnings'].to_dict()
        except Exception:
            pass

    results = []
    success = 0

    for file_path in tqdm(files):
        try:
            filename = os.path.basename(file_path).replace('.csv', '')
            parts = filename.split('_', 1)
            code = parts[0]
            name = parts[1] if len(parts) > 1 else code

            # ── 1. 데이터 로드 ──
            df = pd.read_csv(file_path)
            if 'Date' in df.columns:
                df['Date'] = pd.to_datetime(df['Date'])
                df.set_index('Date', inplace=True)
                df.sort_index(inplace=True)
            df = check_and_fix_columns(df)
            if df is None:
                continue

            # ── 2. 지표 계산 ──
            df = calculate_indicators(df)
            if df is None or len(df) < 60:
                continue

            # ── 3. Regime 계산 ──
            df = add_market_regime(df)
            current_regime = int(df['Regime'].iloc[-1])
            regime_info = detect_regime_transition(df)

            # ── 4. 전략 적용 ──
            df = apply_all_strategies(df, regime=current_regime)
            df = check_exit_signals(df)
            if df is None:
                continue

            # ── 5. Optuna (선택적) ──
            opt_result = None
            if use_optuna:
                opt_result = optimize_with_optuna(df)

            # ── 6. 매물대 분석 ──
            vp = analyze_volume_profile(df, period=120)
            hurdle = check_immediate_hurdle(df, lookback=40)

    # ── 7. 섹터/ETF 매핑 ──
            sec_info = sector_mapper.get(code, {})
            industry = sec_info.get('Industry', '-')
            sector_name = sec_info.get('Sector', 'Unknown')
            
            # GICS 기반 섹터 ETF 조회 (기존 로직 유지 - 백테스트/ETF 참조용)
            sector_etf, sector_kr = get_sector_etf(industry)
            if sector_etf == '-':
                sector_etf, sector_kr = get_sector_etf(sector_name)
            
            current_sec_status = sector_status.get(sector_etf, '-')

            # ✨ HTS 스타일 한국어 직관적 섹터 변환 추가
            hts_sector = get_hts_sector(industry, sector_name)

            # 실적 D-Day
            clean_date, d_day_str = calculate_d_day(earnings_map.get(code, '-'))
            # 레버리지 ETF
            bull, bear = '-', '-'
            if code in etf_mapper:
                bull = etf_mapper[code].get('Bull_ETF', '-')
                bear = etf_mapper[code].get('Bear_ETF', '-')

            # ── 8. 마지막 행 기준 결과 추출 ──
            last = df.iloc[-1]

            # 활성 전략 분석
            active_strats = get_active_strategies(last)
            strat_names = [s['name'] for s in active_strats]
            strat_styles = list(set(s['style'] for s in active_strats))

            # 손절/타겟 계산 (전략 기반 또는 ATR 기반)
            if active_strats:
                levels = calculate_strategy_levels(last, active_strats)
            else:
                levels = calculate_dynamic_levels(df, current_regime)
                if levels is None:
                    levels = {'target': None, 'stop': None, 'target_pct': None, 'stop_pct': None, 'rr_ratio': None}

            # Optuna 결과 오버라이드
            if opt_result:
                opt_target = round(last['Close'] * (1 + opt_result['optimized_tp_pct']), 2)
                opt_stop = round(last['Close'] * (1 - opt_result['optimized_sl_pct']), 2)
                levels['target'] = opt_target
                levels['stop'] = opt_stop
                levels['target_pct'] = round(opt_result['optimized_tp_pct'] * 100, 2)
                levels['stop_pct'] = round(opt_result['optimized_sl_pct'] * 100, 2)

            # ── 매수 패턴 메시지 ──
            buy_msg = []
            if last.get('Candle_Engulfing'):
                buy_msg.append('상승장악')
            if last.get('Vol_Pump'):
                buy_msg.append('거래량폭발')
            if last.get('Candle_Hammer'):
                buy_msg.append('망치형')
            if last.get('MA20_Bounce'):
                buy_msg.append('20일선지지')
            if last.get('Candle_MorningStar'):
                buy_msg.append('모닝스타')
            if last.get('Bullish_Div'):
                buy_msg.append('상승다이버전스')

            # 활성 전략 추가
            for sn in strat_names:
                buy_msg.append(sn)

            # 경고 메시지
            warn_msg = []
            if last.get('MA20_Break'):
                warn_msg.append('20일선붕괴')
            if last.get('Bearish_Div'):
                warn_msg.append('하락다이버전스')
            if last.get('Candle_BearEngulfing'):
                warn_msg.append('하락장악')
            if last.get('Exit_Signal'):
                warn_msg.append('매도신호')

            # ── 최종 결과 구성 ──
            cur = cfg['currency']
            row_data = {
                'Date': last.name.strftime('%Y-%m-%d') if hasattr(last.name, 'strftime') else str(last.name),
                'Code': code,
                'Name': name,
                
                # ✨ HTS 스타일로 교체
                'Sector': hts_sector, 
                'Sector_ETF': sector_etf,
                'Sector_Status': current_sec_status,
                'Industry': industry,
                # 실적
                'Earnings_Date': clean_date,
                'D-Day': d_day_str,
                
                # 가격
                'Close': round(last['Close'], 2),
                
                # Regime
                'Regime': current_regime,
                'Regime_Trend': regime_info['transition'],
                
                # 신호 (기존 호환)
                'Signal_Count': int(last.get('Signal_Count', 0)),
                'Composite_Score': round(float(last.get('Composite_Score', 0)), 2),
                'Buy_Signal': bool(last.get('Final_Buy', False)),
                'Exit_Signal': bool(last.get('Exit_Signal', False)),
                
                # 전략 상세
                'Active_Strategies': ', '.join(strat_names) if strat_names else '-',
                'Style': ', '.join(strat_styles) if strat_styles else '-',
                'Buy_Pattern': ', '.join(buy_msg) if buy_msg else '-',
                'Warning': ', '.join(warn_msg) if warn_msg else '-',
                
                # 레버리지 ETF
                'Bull_ETF': bull,
                'Bear_ETF': bear,
                
                # 타겟/손절 (ATR 기반 동적 계산)
                f'Target({cur})': levels.get('target') or levels.get('target_price'),
                f'Stop({cur})': levels.get('stop') or levels.get('stop_price'),
                'Target(%)': levels.get('target_pct'),
                'Stop(%)': levels.get('stop_pct'),
                'R:R': levels.get('rr_ratio'),
                
                # 매물대
                'Hurdle_Score': round(hurdle['Hurdle_Score'], 3) if hurdle else 999,
                'SR_Ratio': round(hurdle['SR_Ratio'], 2) if hurdle else 0,
                
                # 백테스트 참고
                'BT_WinRate(%)': backtest_stats.get(sector_name, {}).get('Win_Rate(%)', '-'),
                'BT_AvgDuration': backtest_stats.get(sector_name, {}).get('Avg_Duration', '-'),
                
                # 개별 전략 True/False
                'Trend_Up': bool(last.get('Trend_Up', False)),
                'Momentum': bool(last.get('Momentum', False)),
                'Oversold': bool(last.get('Oversold', False)),
                'Vol_Explode': bool(last.get('Vol_Explode', False)),
                'Vol_Pump': bool(last.get('Vol_Pump', False)),
                'Candle_Buy': bool(last.get('Candle_Buy', False)),
                'MA20_Bounce': bool(last.get('MA20_Bounce', False)),
                'MA20_Break': bool(last.get('MA20_Break', False)),
            }
            results.append(row_data)
            success += 1

        except Exception as e:
            # 디버깅 시 주석 해제:
            # print(f"Error {file_path}: {e}")
            continue

    # ── 결과 저장 ──
    if results:
        all_df = pd.DataFrame(results)
        all_df = all_df.drop_duplicates(subset=['Code'], keep='first')
        all_df = all_df.sort_values(
            by=['Buy_Signal', 'Composite_Score', 'Signal_Count'],
            ascending=[False, False, False]
        )

        all_file = f"{cfg['result_dir']}/{market}_{TODAY}_All_Stocks.csv"
        buy_file = f"{cfg['result_dir']}/{market}_{TODAY}_Buy_Signals.csv"
        exit_file = f"{cfg['result_dir']}/{market}_{TODAY}_Exit_Signals.csv"

        all_df.to_csv(all_file, index=False, encoding='utf-8-sig')

        buy_df = all_df[all_df['Buy_Signal'] == True]
        buy_df.to_csv(buy_file, index=False, encoding='utf-8-sig')

        exit_df = all_df[all_df['Exit_Signal'] == True]
        exit_df.to_csv(exit_file, index=False, encoding='utf-8-sig')

        print(f"\n✅ 분석 완료! (총 {success}개 처리)")
        print(f"🔥 매수 포착: {len(buy_df)}개")
        print(f"⚠️ 매도 신호: {len(exit_df)}개")
        print(f"📂 결과: {buy_file}")

        # 텔레그램 전송
        try:
            import telegram_msg
            msg = f"🚀 **[{market}] {TODAY} 분석 완료!**\n\n"
            msg += f"🔥 매수: {len(buy_df)}개 | ⚠️ 매도: {len(exit_df)}개\n"
            if not buy_df.empty:
                top5 = buy_df.head(5)
                msg += "\n**Top 5:**\n"
                for _, r in top5.iterrows():
                    msg += f"• {r['Code']} ({r['Composite_Score']}점) "
                    msg += f"| {r['Active_Strategies']}\n"
            telegram_msg.send_message(msg)
            telegram_msg.send_file(buy_file, caption=f"🔥 {market} 매수 리스트")
        except Exception:
            pass
    else:
        print("❌ 결과 데이터가 없습니다.")


# ============================================================
# CLI 지원
# ============================================================
# if __name__ == '__main__':
#     parser = argparse.ArgumentParser(description='시장 분석 스크립트')
#     parser.add_argument('--market', type=str, default='US_ALL',
#                         choices=['US_ALL', 'KR_ALL'],
#                         help='분석 대상 시장 (US_ALL / KR_ALL)')
#     parser.add_argument('--tickers', nargs='*', default=None,
#                         help='특정 종목만 분석 (예: TSLA NVDA)')
#     parser.add_argument('--optuna', action='store_true',
#                         help='Optuna 최적화 사용')
    
#     # 🔴 수정된 부분: parse_args() 대신 parse_known_args()를 사용합니다.
#     # 이렇게 하면 노트북이 전달하는 --f 인자를 unknown으로 빼고 에러를 내지 않습니다.
#     args, unknown = parser.parse_known_args()
    
#     run_analysis(
#         market=args.market,
#         target_tickers=args.tickers,
#         use_optuna=args.optuna,
#     )



In [5]:
if __name__ == '__main__':
    parser = argparse.ArgumentParser(description='시장 분석 스크립트')
    parser.add_argument('--market', type=str, default='US_ALL',
                        choices=['US_ALL', 'KR_ALL'],
                        help='분석 대상 시장 (US_ALL / KR_ALL)')
    parser.add_argument('--tickers', nargs='*', default=None,
                        help='특정 종목만 분석 (예: TSLA NVDA)')
    parser.add_argument('--optuna', action='store_true',
                        help='Optuna 최적화 사용')
    
    args = parser.parse_args(' --market US_ALL --tickers IMXI FOSL EDRY ASTE ENSG MU SNDK --optuna'.split())
    run_analysis(
        market=args.market,
        target_tickers=args.tickers,
        use_optuna=args.optuna,
    )

🔍 지정된 7개 종목만 분석: ['IMXI', 'FOSL', 'EDRY', 'ASTE', 'ENSG', 'MU', 'SNDK']
🚀 [US_ALL] 분석 시작 (총 7개)
   ⏳ Optuna 최적화 활성화 — 시간이 걸립니다.


100%|██████████| 7/7 [00:02<00:00,  3.41it/s]


✅ 분석 완료! (총 7개 처리)
🔥 매수 포착: 7개
⚠️ 매도 신호: 0개
📂 결과: ./Results/US_ALL/2026-02-23/US_ALL_2026-02-23_Buy_Signals.csv


# mark 2 run 모델
## 거래량 관련된 내용이 들어가 있음. 참조는 readme_v2.1.md 확인

In [9]:
"""
run_analysis.py — 통합 분석 실행 스크립트 (v2.1)
=================================================
[v2.1 핵심 변경]
- Effective_Score 도입 (Technical × Liquidity - Penalty)
- Sector_Status 연동 (sector_analysis.py 결과 반영)
- Money/Price 신호 분리 표시
- VAI, Penalty 컬럼 추가

[사용법]
  python run_analysis.py --market US_ALL
  python run_analysis.py --market US_ALL --tickers TSLA NVDA
  python run_analysis.py --market KR_ALL
  python run_analysis.py --market US_ALL --run-sector   # 섹터 분석 먼저 실행
"""

import pandas as pd
import numpy as np
import os
import sys
import glob
from datetime import datetime
from tqdm import tqdm

# 모듈 임포트
from indicators_v2 import (
    calculate_indicators, check_immediate_hurdle, analyze_volume_profile,
    calculate_dynamic_levels
)
from strategies import (
    apply_all_strategies, check_exit_signals,
    get_active_strategies, calculate_strategy_levels,
    ALL_STRATEGIES, MONEY_STRATEGIES
)
from regime_v2 import add_market_regime, detect_regime_transition, get_trading_params
from sector_map import get_sector_etf, INDUSTRY_TO_SECTOR, GICS_SECTOR_ETF
from sector_analysis import load_sector_status, get_sector_status_for_stock

# ============================================================
# 시장별 설정
# ============================================================

MARKET_CONFIG = {
    'US_ALL': {
        'raw_dir': './Raw_Data/NASDAQ',
        'result_dir': './Results/NASDAQ',
        'etf_file': './ETF_Mappings/etf_full_db.csv',
        'sector_map': INDUSTRY_TO_SECTOR,
        'currency': '$',
        'market_label': 'US',
    },
    'KR_ALL': {
        'raw_dir': './Raw_Data/KR_ALL',
        'result_dir': './Results/KR_ALL',
        'etf_file': './ETF_Mappings/etf_kr_db.csv',
        'sector_map': {},
        'currency': '₩',
        'market_label': 'KR',
    },
}

TODAY = datetime.now().strftime('%Y-%m-%d')


# ============================================================
# 보조 함수
# ============================================================

def load_etf_mappings(etf_file):
    """ETF 매핑 파일 로드"""
    if not os.path.exists(etf_file):
        return {}
    try:
        df = pd.read_csv(etf_file)
        mappings = {}
        for _, row in df.iterrows():
            code = str(row.get('Code', row.get('Ticker', ''))).strip()
            if code:
                mappings[code] = {
                    'Bull_ETF': row.get('Bull_ETF', '-'),
                    'Bear_ETF': row.get('Bear_ETF', '-'),
                }
        return mappings
    except Exception:
        return {}


def load_earnings_calendar():
    """어닝 캘린더 로드"""
    patterns = ['./Raw_Data/Earnings*.csv', './Results/Earnings*.csv']
    for p in patterns:
        files = glob.glob(p)
        if files:
            try:
                df = pd.read_csv(files[0])
                cal = {}
                for _, row in df.iterrows():
                    code = str(row.get('Code', row.get('Ticker', ''))).strip()
                    if code:
                        cal[code] = str(row.get('Earnings_Date', '-'))
                return cal
            except Exception:
                pass
    return {}


def load_backtest_stats():
    """백테스트 결과 로드"""
    patterns = ['./Results/*backtest*.csv', './Results/*Backtest*.csv']
    for p in patterns:
        files = glob.glob(p)
        if files:
            try:
                df = pd.read_csv(files[0])
                stats = {}
                for _, row in df.iterrows():
                    code = str(row.get('Code', row.get('Ticker', ''))).strip()
                    if code:
                        stats[code] = {
                            'WinRate': row.get('WinRate(%)', '-'),
                            'AvgDuration': row.get('AvgDuration', '-'),
                        }
                return stats
            except Exception:
                pass
    return {}


def calc_earnings_dday(earnings_date_str):
    """D-Day 계산"""
    if not earnings_date_str or earnings_date_str == '-':
        return '-'
    try:
        edate = pd.to_datetime(earnings_date_str)
        diff = (edate - pd.Timestamp(TODAY)).days
        if diff < 0:
            return '지남'
        return f'D-{diff}'
    except Exception:
        return '-'


# ============================================================
# 메인 분석
# ============================================================

def run_analysis(market='US_ALL', tickers=None, run_sector=False, use_optuna=False):
    """
    메인 분석 실행
    
    Parameters:
    -----------
    market : str — 'US_ALL' or 'KR_ALL'
    tickers : list or None — 특정 종목만 분석
    run_sector : bool — 섹터 분석 먼저 실행 여부
    use_optuna : bool — Optuna 최적화 사용 여부
    """
    config = MARKET_CONFIG.get(market)
    if not config:
        print(f"❌ 지원하지 않는 시장: {market}")
        return

    print(f"{'='*60}")
    print(f"📊 [{market}] 분석 시작 — {TODAY}")
    print(f"{'='*60}")

    # ── 섹터 분석 실행 (옵션) ──
    if run_sector:
        print("\n🌍 섹터 분석 먼저 실행...")
        try:
            from sector_analysis import analyze_sectors
            analyze_sectors(market=config['market_label'])
        except Exception as e:
            print(f"  ⚠️ 섹터 분석 오류: {e}")

    # ── 섹터 상태 로드 ──
    sector_status_map = load_sector_status()
    if sector_status_map:
        print(f"  ✅ 섹터 상태 로드: {len(sector_status_map)}개")
    else:
        print("  ⚠️ 섹터 상태 없음 (--run-sector로 먼저 실행 권장)")

    # ── 데이터 소스 ──
    raw_dir = config['raw_dir']
    if not os.path.exists(raw_dir):
        print(f"❌ 데이터 폴더 없음: {raw_dir}")
        return

    etf_map = load_etf_mappings(config['etf_file'])
    earnings_cal = load_earnings_calendar()
    bt_stats = load_backtest_stats()
    currency = config['currency']

    # ── CSV 파일 목록 ──
    if tickers:
        csv_files = []
        for t in tickers:
            pattern = f"{raw_dir}/*{t}*.csv"
            csv_files.extend(glob.glob(pattern))
    else:
        csv_files = glob.glob(f"{raw_dir}/*.csv")

    if not csv_files:
        print(f"❌ CSV 파일 없음: {raw_dir}")
        return

    print(f"  📁 대상 종목: {len(csv_files)}개\n")

    # ── 결과 수집 ──
    all_results = []
    buy_results = []
    exit_results = []

    for csv_path in tqdm(csv_files, desc="분석 중"):
        try:
            code = os.path.basename(csv_path).replace('.csv', '').split('_')[0]
            df = pd.read_csv(csv_path)

            # 지표 계산
            df = calculate_indicators(df)
            if df is None or len(df) < 60:
                continue

            # Regime
            df = add_market_regime(df)
            regime_info = detect_regime_transition(df)

            # 전략 적용
            df = apply_all_strategies(df, regime=regime_info['current'])
            df = check_exit_signals(df)

            last = df.iloc[-1]

            # 섹터 정보
            industry = str(last.get('Industry', '-'))
            sector_etf_ticker, sector_name_kr = get_sector_etf(code, industry)
            sector_status = get_sector_status_for_stock(sector_etf_ticker, sector_status_map)

            # ETF 매핑
            etf_info = etf_map.get(code, {'Bull_ETF': '-', 'Bear_ETF': '-'})

            # 어닝
            earnings_date = earnings_cal.get(code, '-')
            dday = calc_earnings_dday(earnings_date)

            # 활성 전략
            active = get_active_strategies(last)
            active_names = ', '.join([s['name'] for s in active]) if active else '-'
            styles = ', '.join(sorted(set(s['style'] for s in active))) if active else '-'
            money_names = [s['name'] for s in active if s['is_money']]

            # 손절/타겟 계산
            if active:
                levels = calculate_strategy_levels(last, active)
            else:
                dl = calculate_dynamic_levels(df, regime=regime_info['current'])
                levels = {
                    'target': dl['target_price'] if dl else None,
                    'stop': dl['stop_price'] if dl else None,
                    'target_pct': dl['target_pct'] if dl else None,
                    'stop_pct': dl['stop_pct'] if dl else None,
                    'rr_ratio': dl['rr_ratio'] if dl else None,
                } if dl else {'target': None, 'stop': None, 'target_pct': None, 'stop_pct': None, 'rr_ratio': None}

            # 매물대
            hurdle = check_immediate_hurdle(df, lookback=40)
            vp = analyze_volume_profile(df, period=120)

            # 백테스트
            bt = bt_stats.get(code, {'WinRate': '-', 'AvgDuration': '-'})

            # 경고
            warnings_list = []
            if last.get('Warning_Count', 0) >= 2:
                warnings_list.append('경고2+')
            if last.get('vai_spike_only', False):
                warnings_list.append('단발거래량')
            if last.get('upper_wick_ratio', 0) > 0.6:
                warnings_list.append('윗꼬리')
            if last.get('Penalty', 0) >= 1.5:
                warnings_list.append(f"P{last['Penalty']:.1f}")
            warning_str = ', '.join(warnings_list) if warnings_list else '-'

            # 레거시 전략 목록 (Buy_Pattern용)
            legacy_signals = []
            for sig_name in ['Trend_Up', 'Momentum', 'Oversold', 'Vol_Explode', 'Vol_Pump', 'Candle_Buy']:
                if last.get(sig_name, False):
                    legacy_signals.append(sig_name)

            # 결과 행
            row_data = {
                'Date': df.index[-1].strftime('%Y-%m-%d') if hasattr(df.index[-1], 'strftime') else str(df.index[-1]),
                'Code': code,
                'Name': last.get('Name', code),
                'Sector': sector_name_kr if sector_name_kr != '-' else industry,
                'Sector_ETF': sector_etf_ticker if sector_etf_ticker else '-',
                'Sector_Status': sector_status,
                'Industry': industry,
                'Earnings_Date': earnings_date,
                'D-Day': dday,
                'Close': round(float(last['Close']), 2),
                'Regime': regime_info['current'],
                'Regime_Trend': regime_info['transition'],
                'Signal_Count': int(last.get('Signal_Count', 0)),
                # ★ 핵심 새 컬럼들
                'Technical_Score': round(float(last.get('Technical_Score', 0)), 2),
                'Money_Score': round(float(last.get('Money_Score', 0)), 2),
                'Price_Score': round(float(last.get('Price_Score', 0)), 2),
                'Liquidity_Score': round(float(last.get('liquidity_score', 0.5)), 2),
                'Penalty': round(float(last.get('Penalty', 0)), 2),
                'Effective_Score': round(float(last.get('Effective_Score', 0)), 2),
                # 기존 호환
                'Composite_Score': round(float(last.get('Effective_Score', 0)), 2),
                'Buy_Signal': bool(last.get('Final_Buy', last.get('Buy_Signal', False))),
                'Exit_Signal': bool(last.get('Exit_Signal', False)),
                'Active_Strategies': active_names,
                'Money_Strategies': ', '.join(money_names) if money_names else '-',
                'Style': styles,
                'Buy_Pattern': ', '.join(legacy_signals) if legacy_signals else '-',
                'Warning': warning_str,
                # ★ VAI 관련
                'VAI_Stage1': round(float(last.get('vai_stage1', 0)), 2),
                'VAI_Stage2': round(float(last.get('vai_stage2', 0)), 2),
                'VAI_Sustained': bool(last.get('vai_sustained', False)),
                'RVOL': round(float(last.get('rvol', 0)), 2),
                # ETF
                'Bull_ETF': etf_info.get('Bull_ETF', '-'),
                'Bear_ETF': etf_info.get('Bear_ETF', '-'),
                # 손절/타겟
                f'Target({currency})': levels.get('target', '-'),
                f'Stop({currency})': levels.get('stop', '-'),
                'Target(%)': levels.get('target_pct', '-'),
                'Stop(%)': levels.get('stop_pct', '-'),
                'R:R': levels.get('rr_ratio', '-'),
                # 매물대
                'Hurdle_Score': round(hurdle['Hurdle_Score'], 3) if hurdle else 999,
                'SR_Ratio': round(hurdle['SR_Ratio'], 2) if hurdle else 0,
                # 백테스트
                'BT_WinRate(%)': bt.get('WinRate', '-'),
                'BT_AvgDuration': bt.get('AvgDuration', '-'),
                # 캔들 페널티
                'Upper_Wick_Ratio': round(float(last.get('upper_wick_ratio', 0)), 3),
                'High_Close_Drop(%)': round(float(last.get('high_close_drop', 0)), 2),
                # 기존 불리언
                'Trend_Up': bool(last.get('Trend_Up', False)),
                'Momentum': bool(last.get('Momentum', False)),
                'Oversold': bool(last.get('Oversold', False)),
                'Vol_Explode': bool(last.get('Vol_Explode', False)),
                'Vol_Pump': bool(last.get('Vol_Pump', False)),
                'Candle_Buy': bool(last.get('Candle_Buy', False)),
            }

            all_results.append(row_data)

            if row_data['Buy_Signal']:
                buy_results.append(row_data)
            if row_data['Exit_Signal']:
                exit_results.append(row_data)

        except Exception as e:
            continue

    # ── 결과 저장 ──
    result_dir = config['result_dir']
    os.makedirs(result_dir, exist_ok=True)

    if all_results:
        df_all = pd.DataFrame(all_results)
        df_all = df_all.sort_values('Effective_Score', ascending=False)
        all_path = f"{result_dir}/{market}_{TODAY}_All_Stocks.csv"
        df_all.to_csv(all_path, index=False, encoding='utf-8-sig')
        print(f"\n📄 전체 결과: {all_path} ({len(df_all)}개)")

    if buy_results:
        df_buy = pd.DataFrame(buy_results)
        df_buy = df_buy.sort_values('Effective_Score', ascending=False)
        buy_path = f"{result_dir}/{market}_{TODAY}_Buy_Signals.csv"
        df_buy.to_csv(buy_path, index=False, encoding='utf-8-sig')
        print(f"🟢 매수 신호: {buy_path} ({len(df_buy)}개)")

        # 상위 5개 요약
        print(f"\n{'='*60}")
        print(f"🏆 TOP 매수 후보")
        print(f"{'='*60}")
        for i, (_, r) in enumerate(df_buy.head(5).iterrows()):
            money_tag = '💰' if r['Money_Score'] > 0 else '📊'
            sector_tag = f"[{r['Sector_Status']}]" if r['Sector_Status'] != '-' else ''
            print(f"  {i+1}. {money_tag} {r['Code']} ({r['Name']})")
            print(f"     Effective: {r['Effective_Score']} "
                  f"(Tech:{r['Technical_Score']} × Liq:{r['Liquidity_Score']} - P:{r['Penalty']})")
            print(f"     Money: {r['Money_Score']} | Price: {r['Price_Score']} "
                  f"| R{r['Regime']}({r['Regime_Trend']})")
            print(f"     VAI: {r['VAI_Stage1']:.1f}/{r['VAI_Stage2']:.1f} "
                  f"{'⚡지속' if r['VAI_Sustained'] else ''} | RVOL: {r['RVOL']:.1f}x")
            print(f"     Sector: {r['Sector']} {sector_tag}")
            print(f"     전략: {r['Active_Strategies']}")
            if r['Warning'] != '-':
                print(f"     ⚠️ {r['Warning']}")
            print()
        print(f"{'='*60}")

        # 텔레그램
        _send_telegram(df_buy, market, currency)

    else:
        print("\n⚪ 매수 신호 없음")

    if exit_results:
        df_exit = pd.DataFrame(exit_results)
        exit_path = f"{result_dir}/{market}_{TODAY}_Exit_Signals.csv"
        df_exit.to_csv(exit_path, index=False, encoding='utf-8-sig')
        print(f"🔴 매도 신호: {exit_path} ({len(df_exit)}개)")

    print(f"\n✅ [{market}] 분석 완료!")


def _send_telegram(df_buy, market, currency):
    """텔레그램 매수 알림 전송"""
    try:
        import telegram_msg

        msg = f"📊 **[{market}] {TODAY} 매수 신호**\n\n"
        msg += f"총 {len(df_buy)}개 | 실효점수 기반\n\n"

        for i, (_, r) in enumerate(df_buy.head(5).iterrows()):
            money_tag = '💰' if r['Money_Score'] > 0 else '📊'
            msg += f"{i+1}. {money_tag} **{r['Code']}** ({r['Name']})\n"
            msg += f"   E:{r['Effective_Score']} (T:{r['Technical_Score']}×L:{r['Liquidity_Score']}-P:{r['Penalty']})\n"
            msg += f"   R{r['Regime']} | {r['Style']} | {r['Active_Strategies']}\n"
            if r['Sector_Status'] != '-':
                msg += f"   섹터: {r['Sector']} [{r['Sector_Status']}]\n"
            target_val = r.get(f'Target({currency})', '-')
            stop_val = r.get(f'Stop({currency})', '-')
            msg += f"   Target: {currency}{target_val} | Stop: {currency}{stop_val}\n\n"

        telegram_msg.send_message(msg)

    except Exception:
        pass




In [ ]:
if __name__ == '__main__':
    parser = argparse.ArgumentParser(description='시장 분석 스크립트')
    parser.add_argument('--market', type=str, default='US_ALL',
                        choices=['US_ALL', 'KR_ALL'],
                        help='분석 대상 시장 (US_ALL / KR_ALL)')
    parser.add_argument('--tickers', nargs='*', default=None,
                        help='특정 종목만 분석 (예: TSLA NVDA)')
    parser.add_argument('--optuna', action='store_true',
                        help='Optuna 최적화 사용')
    
    # args = parser.parse_args(' --market US_ALL --tickers IMXI FOSL EDRY ASTE ENSG MU SNDK --optuna'.split())
    # args = parser.parse_args(' --market KR_ALL' .split())
    run_analysis(market='KR_ALL', run_sector=True)
    # run_analysis(market='KR_ALL')

    # run_analysis(
    #     market=args.market,
    #     target_tickers=args.tickers,
    #     use_optuna=args.optuna,
    # )

📊 [KR_ALL] 분석 시작 — 2026-02-24

🌍 섹터 분석 먼저 실행...
🚀 [섹터 분석] KR 주요 8개 산업군 진단 시작...



100%|██████████| 8/8 [00:03<00:00,  2.33it/s]



✅ 섹터 분석 완료!
  🟢 Good (상승/매수): 8개
  🔴 Bad  (하락/관망): 0개
  📂 Good 차트: ./Results/Sector_Analysis/2026-02-24/Good_Sectors
  📂 Bad  차트: ./Results/Sector_Analysis/2026-02-24/Bad_Sectors
  📄 요약 CSV: ./Results/Sector_Analysis/2026-02-24/Sector_Status_Summary.csv

  🟢 [091180] KODEX 자동차                 | 상승추세   | R3 | RVOL 0.5x | ADX 39
  🟢 [305720] KODEX 2차전지                | 상승추세   | R4 | RVOL 1.2x | ADX 32
  🟢 [091160] KODEX 반도체                 | 상승추세   | R3 | RVOL 0.9x | ADX 42
  🟢 [117700] KODEX 건설                  | 상승추세   | R3 | RVOL 0.9x | ADX 56
  🟢 [117460] KODEX 에너지화학               | 상승추세   | R4 | RVOL 1.4x | ADX 31
  🟢 [266370] KODEX 바이오                 | 상승추세   | R3 | RVOL 0.7x | ADX 39
  🟢 [244580] KODEX 바이오(대형)             | 상승추세   | R3 | RVOL 1.2x | ADX 14
  🟢 [091170] KODEX 은행                  | 상승추세   | R3 | RVOL 0.8x | ADX 53
  ✅ 섹터 상태 로드: 8개
  📁 대상 종목: 213개



분석 중: 100%|██████████| 213/213 [00:07<00:00, 26.66it/s]


📄 전체 결과: ./Results/KR_ALL/KR_ALL_2026-02-24_All_Stocks.csv (213개)
🟢 매수 신호: ./Results/KR_ALL/KR_ALL_2026-02-24_Buy_Signals.csv (19개)

🏆 TOP 매수 후보
  1. 💰 096770 (096770)
     Effective: 10.4 (Tech:8.0 × Liq:1.3 - P:0.0)
     Money: 5.5 | Price: 2.5 | R4(improving)
     VAI: 1.8/1.4 ⚡지속 | RVOL: 2.4x
     Sector: - 
     전략: VolBreakout, VolPumpSustained, MoneyFlowSurge, SqueezeBreakout, IchimokuTrend

  2. 💰 006650 (006650)
     Effective: 9.75 (Tech:7.5 × Liq:1.3 - P:0.0)
     Money: 5.5 | Price: 2.0 | R3(stable)
     VAI: 1.7/1.3 ⚡지속 | RVOL: 2.1x
     Sector: - 
     전략: VolBreakout, VolPumpSustained, MoneyFlowSurge, GapMomentum, IchimokuTrend

  3. 💰 006400 (006400)
     Effective: 6.8 (Tech:6.0 × Liq:1.3 - P:1.0)
     Money: 1.5 | Price: 4.5 | R4(improving)
     VAI: 2.8/0.7  | RVOL: 2.0x
     Sector: - 
     전략: MoneyFlowSurge, GapMomentum, MACDReversal, IchimokuTrend, TrendContinuation
     ⚠️ 단발거래량

  4. 💰 005440 (005440)
     Effective: 5.35 (Tech:4.5 × Liq:1.3 - P:0.5)
     Mone

: 

In [2]:
# ============================================================
# CLI
# ============================================================

if __name__ == '__main__':
    import argparse
    parser = argparse.ArgumentParser(description='통합 분석 실행')
    parser.add_argument('--market', type=str, default='US_ALL',
                        choices=['US_ALL', 'KR_ALL'], help='시장 선택')
    parser.add_argument('--tickers', nargs='+', default=None,
                        help='특정 종목만 분석 (예: TSLA NVDA)')
    parser.add_argument('--run-sector', action='store_true',
                        help='섹터 분석 먼저 실행')
    parser.add_argument('--optuna', action='store_true',
                        help='Optuna 최적화 사용')
    args = parser.parse_args()

run_analysis(market='KR_ALL')



usage: ipykernel_launcher.py [-h] [--market {US_ALL,KR_ALL}]
                             [--tickers TICKERS [TICKERS ...]] [--run-sector]
                             [--optuna]
ipykernel_launcher.py: error: unrecognized arguments: --f=c:\Users\User\AppData\Roaming\jupyter\runtime\kernel-v39813465c8cf13e9e953fad50b90ac27fce528160.json


SystemExit: 2